# Market & Fundamental Data Definitive

Definitive research notebook using the repository-wide database manifest and universal notebook standards.


## 0. Setup & Config

Configure dependencies, experiment parameters, database paths, logging, and plotting style before any data is touched.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

The setup cell makes the data-source decision explicit: local cache, Google Drive, or API. Economically, reproducible configuration is what turns a promising data signal into a testable research claim.


In [ ]:
import subprocess
import sys
from pathlib import Path

DEPENDENCIES = [
    "gdown",
    "matplotlib",
    "numpy",
    "pandas",
    "pyarrow",
    "requests",
    "scikit-learn",
    "seaborn",
    "statsmodels",
    "yfinance",
]
for package in DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)

import logging
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "machine-learning-for-trading" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.definitive_data_loader import (
    domain_frame,
    load_dataset,
    load_dotenv_files,
    load_manifest,
    manifest_frame,
    source_summary,
    synthetic_fallback,
    write_table,
)

OUTPUT_DIR = Path("output")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
LOG_DIR = OUTPUT_DIR / "logs"
for folder in [FIGURE_DIR, TABLE_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("definitive_notebook")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "experiment.log")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

np.random.seed(42)

COLORS = {
    "primary": "#01696f",
    "accent": "#da7101",
    "q1": "#c0392b",
    "neutral": "#7a7974",
    "bg": "#f7f6f2",
    "blue": "#006494",
    "gold": "#d19900",
    "purple": "#7a39bb",
}
MODEL_COLORS = {
    "linear": COLORS["blue"],
    "ridge": COLORS["primary"],
    "random_forest": COLORS["accent"],
    "baseline": COLORS["neutral"],
}
plt.rcParams.update({"figure.facecolor": COLORS["bg"], "axes.facecolor": COLORS["bg"], "savefig.dpi": 180})
sns.set_theme(style="whitegrid", font_scale=1.05)

EXPERIMENT = {
    "start_date": "2018-01-01",
    "end_date": "2026-05-12",
    "target": "forward_return_21d",
    "horizons": [1, 5, 21],
    "test_start": "2023-01-01",
    "embargo": 21,
    "n_quantiles": 5,
    "cost_bps": 10.0,
    "models": ["linear", "ridge", "random_forest"],
    "run_ablation": True,
    "run_backtest": True,
    "save_figures": True,
    "feature_blocks": ["controls", "source_quality", "returns", "alternative"],
    "data_source": "auto",
    "min_train_size": 60,
    "min_test_size": 5,
    "rolling_windows": [5, 21, 63],
    "winsor_limits": [0.01, 0.99],
    "n_estimators": 100,
    "random_state": 42,
    "top_quantile": 0.8,
    "bottom_quantile": 0.2,
    "structural_breaks": ["2020-03-16", "2022-02-24"],
    "figure_dpi": 180,
}

UNIVERSE = {
    "assets": ["AAPL", "SPY", "MSFT", "NVDA", "TSLA"],
    "database_root_drive": "https://drive.google.com/drive/folders/11xTU7S06NBFSEtvHM4CFLab7uqRenlI1?usp=drive_link",
    "local_database_root": str(REPO_ROOT / "data"),
}

load_dotenv_files()
MANIFEST = load_manifest()
manifest_df = manifest_frame(MANIFEST)
domains_df = domain_frame(MANIFEST)
write_table(manifest_df, TABLE_DIR / "Table_0_database_manifest.csv")
print(manifest_df[["dataset", "domain", "kind", "local_path", "drive_url"]].to_string(index=False))

SELECTED_DATASETS = ['data_inventory', 'data_domains', 'api_providers', 'api_provider_registry', 'aapl_open_equity', 'sp500_constituents']
PRIMARY_DATASET_ORDER = ['aapl_open_equity', 'api_provider_registry', 'data_inventory']


## 1. Data Ingestion

Load the definitive database manifest and then load project datasets from local files, Google Drive, or registered APIs.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

The same research question can be tested against cached evidence or refreshed APIs. A source summary keeps provenance visible, which matters because data lineage is part of financial model risk.


In [ ]:
def fetch_aapl_from_yfinance() -> pd.DataFrame:
    import yfinance as yf
    data = yf.download("AAPL", start=EXPERIMENT["start_date"], end=EXPERIMENT["end_date"], progress=False, auto_adjust=True)
    data = data.reset_index()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [col[0].lower() if col[0] else col[1].lower() for col in data.columns]
    data.columns = [str(col).lower().replace(" ", "_") for col in data.columns]
    return data

def fetch_polymarket_public() -> pd.DataFrame:
    response = requests.get("https://gamma-api.polymarket.com/markets", params={"limit": 100}, timeout=20)
    response.raise_for_status()
    return pd.DataFrame(response.json())

API_FETCHERS = {
    "aapl_open_equity": fetch_aapl_from_yfinance,
    "polymarket_feature_snapshot": fetch_polymarket_public,
}

DATASET_NAMES = SELECTED_DATASETS
loaded = {}
load_results = []
for dataset_name in DATASET_NAMES:
    try:
        df, result = load_dataset(dataset_name, source=EXPERIMENT["data_source"], manifest=MANIFEST, api_fetchers=API_FETCHERS, logger=logger)
    except Exception as exc:
        logger.warning("%s unavailable from real sources: %s", dataset_name, exc)
        df, result = synthetic_fallback(dataset_name, periods=EXPERIMENT["rolling_windows"][-1] * 8, seed=EXPERIMENT["random_state"])
        logger.warning("%s uses synthetic fallback columns marked with _synthetic", dataset_name)
    loaded[dataset_name] = df
    load_results.append(result)

source_table = source_summary(load_results)
write_table(source_table, TABLE_DIR / "Table_1_data_source_summary.csv")
print("DATA SOURCE SUMMARY")
print(source_table.to_string(index=False))


## 2. Cleaning & Alignment

Standardize dates, remove duplicate observations, and align tables on a common research index where available.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Financial and alternative datasets are often asynchronous. Alignment prevents stale observations from masquerading as predictive information.


In [ ]:
def normalize_dates(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    date_candidates = [col for col in data.columns if str(col).lower() in {"date", "datetime", "timestamp"}]
    if date_candidates:
        date_col = date_candidates[0]
        data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
        data = data.dropna(subset=[date_col]).sort_values(date_col).drop_duplicates(subset=[date_col])
        data = data.rename(columns={date_col: "date"})
    return data

cleaned = {name: normalize_dates(df) for name, df in loaded.items()}
cleaning_rows = []
for name, df in cleaned.items():
    cleaning_rows.append({"dataset": name, "N": len(df), "columns": len(df.columns), "duplicate_rows": int(df.duplicated().sum()), "missing_cells": int(df.isna().sum().sum())})
cleaning_table = pd.DataFrame(cleaning_rows)
write_table(cleaning_table, TABLE_DIR / "Table_2_cleaning_alignment.csv")
print(cleaning_table.to_string(index=False))


## 3. Feature Engineering

Build lagged, rolling, and source-quality features grouped into named feature blocks.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Features encode hypotheses about market behavior. Every lagged or rolling transformation uses past information only so the notebook keeps zero look-ahead tolerance.


In [ ]:
def get_primary_panel(cleaned: dict[str, pd.DataFrame]) -> pd.DataFrame:
    for preferred in PRIMARY_DATASET_ORDER:
        if preferred in cleaned:
            return cleaned[preferred].copy()
    return next(iter(cleaned.values())).copy()

panel = get_primary_panel(cleaned)
if "date" not in panel.columns:
    panel = panel.reset_index(drop=True)
    panel["date"] = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=len(panel))

numeric_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    fallback, fallback_result = synthetic_fallback("numeric_panel", periods=EXPERIMENT["rolling_windows"][-1] * 8, seed=EXPERIMENT["random_state"])
    logger.warning("Primary panel has no numeric columns; using synthetic numeric panel")
    panel = fallback
    numeric_cols = panel.select_dtypes(include=[np.number]).columns.tolist()

base_col = numeric_cols[0]
panel = panel.sort_values("date").reset_index(drop=True)
if "return" not in base_col.lower() and panel[base_col].gt(0).all():
    panel["base_return"] = panel[base_col].pct_change().shift(1)
else:
    panel["base_return"] = panel[base_col].shift(1)

def winsorize_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    data = df.copy()
    low_q, high_q = EXPERIMENT["winsor_limits"]
    for col in columns:
        low, high = data[col].quantile([low_q, high_q])
        data[f"{col}_winsor"] = data[col].clip(low, high)
    return data

def build_controls_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["control_trend"] = np.arange(len(data)) / max(len(data), 1)
    data["control_dayofweek"] = pd.to_datetime(data["date"]).dt.dayofweek
    return data

def build_source_quality_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["source_missing_share"] = data.isna().mean(axis=1).shift(1)
    data["source_numeric_count"] = data.select_dtypes(include=[np.number]).notna().sum(axis=1).shift(1)
    return data

def build_returns_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    for window in EXPERIMENT["rolling_windows"]:
        data[f"return_mean_{window}"] = data["base_return"].rolling(window).mean().shift(1)
        data[f"return_vol_{window}"] = data["base_return"].rolling(window).std().shift(1)
    return data

def build_alternative_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["alternative_attention_proxy"] = data.select_dtypes(include=[np.number]).abs().mean(axis=1).rolling(EXPERIMENT["rolling_windows"][0]).mean().shift(1)
    return data

FEATURE_BUILDERS = {
    "controls": build_controls_features,
    "source_quality": build_source_quality_features,
    "returns": build_returns_features,
    "alternative": build_alternative_features,
}

features = panel.copy()
for block in EXPERIMENT["feature_blocks"]:
    features = FEATURE_BUILDERS[block](features)

feature_cols_raw = [col for col in features.select_dtypes(include=[np.number]).columns if col not in {base_col}]
features = winsorize_columns(features, feature_cols_raw)
feature_cols = [col for col in features.columns if col.endswith("_winsor")]

missing_rows = []
for block in EXPERIMENT["feature_blocks"]:
    block_cols = [col for col in feature_cols if block.split("_")[0] in col or col.startswith(block)]
    if not block_cols and block == "controls":
        block_cols = [col for col in feature_cols if col.startswith("control")]
    for col in block_cols:
        missing_rows.append({"block": block, "feature": col, "N": len(features), "missing_pct": features[col].isna().mean() * 100})
missingness_table = pd.DataFrame(missing_rows) if missing_rows else pd.DataFrame(columns=["block", "feature", "N", "missing_pct"])
write_table(missingness_table, TABLE_DIR / "Table_feature_missingness.csv")
print(missingness_table.round(2).to_string(index=False))


## 4. Targets & Labels

Create forward targets and quantile labels for supervised learning or scoring tasks.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Targets translate the research objective into measurable outcomes. Forward labels are separated from features so performance reflects prediction rather than leakage.


In [ ]:
horizon = EXPERIMENT["horizons"][-1]
features[EXPERIMENT["target"]] = features["base_return"].shift(-horizon)
valid_target = features[EXPERIMENT["target"]].dropna()
if len(valid_target) >= EXPERIMENT["n_quantiles"]:
    features["target_quantile"] = pd.qcut(features[EXPERIMENT["target"]], q=EXPERIMENT["n_quantiles"], labels=False, duplicates="drop")
else:
    features["target_quantile"] = np.nan
label_table = features[["date", EXPERIMENT["target"], "target_quantile"]].dropna().copy()
write_table(label_table, TABLE_DIR / "Table_4_targets_labels.csv")
print(label_table.tail(10).round(4).to_string(index=False))


## 5. Descriptive Stats

Summarize sample size, missingness, and core distributional properties for the assembled research panel.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Descriptive diagnostics catch sample problems early. In markets, outliers and missingness often reveal microstructure, listing, or provider artifacts.


In [ ]:
usable_feature_cols = [col for col in feature_cols if features[col].notna().mean() >= 0.5]
analysis_cols = usable_feature_cols + [EXPERIMENT["target"]]
model_data = features[["date"] + analysis_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
if len(model_data) < EXPERIMENT["min_train_size"]:
    logger.warning("Model panel too small after alignment; creating synthetic modelling fallback with _synthetic columns")
    periods = EXPERIMENT["rolling_windows"][-1] * 8
    rng = np.random.default_rng(EXPERIMENT["random_state"])
    model_data = pd.DataFrame({
        "date": pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=periods),
        "control_winsor_synthetic": rng.normal(0, 1, periods),
        "signal_winsor_synthetic": rng.normal(0, 1, periods),
        "source_quality_winsor_synthetic": rng.uniform(0, 1, periods),
    })
    model_data[EXPERIMENT["target"]] = 0.002 * model_data["signal_winsor_synthetic"] + rng.normal(0, 0.02, periods)
    feature_cols = ["control_winsor_synthetic", "signal_winsor_synthetic", "source_quality_winsor_synthetic"]
    analysis_cols = feature_cols + [EXPERIMENT["target"]]
stats_table = model_data[analysis_cols].describe().T.reset_index().rename(columns={"index": "variable", "count": "N"})
write_table(stats_table, TABLE_DIR / "Table_5_descriptive_stats.csv")
print(stats_table.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
model_data[EXPERIMENT["target"]].hist(ax=ax, bins=30, color=COLORS["primary"])
ax.axvline(0, color="black", linestyle="--", lw=0.8, label="zero")
ax.set_title("Target Distribution", fontweight="bold")
ax.set_xlabel("Forward return (decimal)")
ax.set_ylabel("Observations (count)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_5_target_distribution.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 6. Exploratory / Event Study

Inspect cumulative behavior and simple event-style differences before fitting complex models.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Exploration is a plausibility check. If simple plots contradict the hypothesized mechanism, more flexible models usually amplify confusion rather than solve it.


In [ ]:
explore = model_data[["date", EXPERIMENT["target"]]].copy()
explore["cumulative_target"] = (1 + explore[EXPERIMENT["target"]]).cumprod() - 1
explore_table = explore.tail(20)
write_table(explore_table, TABLE_DIR / "Table_6_exploratory_event_study.csv")
print(explore_table.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(explore["date"], explore["cumulative_target"], color=COLORS["primary"], label="cumulative target")
ax.axhline(0, color="black", linestyle="--", lw=0.8)
for break_date in EXPERIMENT["structural_breaks"]:
    ax.axvline(pd.Timestamp(break_date), color=COLORS["accent"], linestyle="--", lw=0.8, label=f"break {break_date}")
ax.set_title("Exploratory Cumulative Target", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative return (decimal)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_6_exploratory_cumulative_target.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 7. Single-Factor Diagnostics

Evaluate each engineered feature one at a time using correlation and quantile spread diagnostics.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Single-factor tests expose whether a feature has standalone economic content. Weak individual signals can still matter, but diagnostics help avoid blind feature accumulation.


In [ ]:
diagnostics = []
for col in feature_cols:
    corr = model_data[[col, EXPERIMENT["target"]]].corr().iloc[0, 1]
    diagnostics.append({"feature": col, "N": model_data[[col, EXPERIMENT["target"]]].dropna().shape[0], "corr_with_target": corr})
diag_table = pd.DataFrame(diagnostics).sort_values("corr_with_target", key=lambda s: s.abs(), ascending=False)
write_table(diag_table, TABLE_DIR / "Table_7_single_factor_diagnostics.csv")
print(diag_table.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
plot_diag = diag_table.head(15).sort_values("corr_with_target")
ax.barh(plot_diag["feature"], plot_diag["corr_with_target"], color=COLORS["blue"])
ax.axvline(0, color="black", linestyle="--", lw=0.8)
ax.set_title("Single-Factor Correlations", fontweight="bold")
ax.set_xlabel("Correlation with target (decimal)")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_7_single_factor_diagnostics.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 8. Statistical Models (regressions / econometrics)

Estimate fixed, interpretable baseline regressions before machine learning models.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Econometric baselines set a transparent hurdle. Coefficients and standard errors help distinguish directional signal from unstable noise.


In [ ]:
X = model_data[feature_cols]
y = model_data[EXPERIMENT["target"]]
X_const = sm.add_constant(X, has_constant="add")
ols = sm.OLS(y, X_const).fit()
reg_table = pd.DataFrame({"feature": ols.params.index, "coef": ols.params.values, "t_stat": ols.tvalues.values, "p_value": ols.pvalues.values})
reg_table["N"] = int(ols.nobs)
write_table(reg_table, TABLE_DIR / "Table_8_statistical_models.csv")
print(reg_table.round(4).to_string(index=False))


## 9. ML Walk-Forward

Run chronological walk-forward models with embargo and train-only scaling.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Walk-forward evaluation mirrors the research-to-live sequence. Training only on the past is non-negotiable for trading systems.


In [ ]:
def embargo_train(train_df: pd.DataFrame, test_dates: pd.Series) -> pd.DataFrame:
    min_test = test_dates.min()
    embargo_days = pd.Timedelta(days=EXPERIMENT["embargo"])
    return train_df[train_df["date"] < min_test - embargo_days]

def walk_forward(data: pd.DataFrame, active_features: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    folds = []
    preds = []
    data = data.sort_values("date").copy()
    test_months = data.loc[data["date"] >= pd.Timestamp(EXPERIMENT["test_start"]), "date"].dt.to_period("M").unique()
    models = {
        "linear": LinearRegression(),
        "ridge": Ridge(alpha=1.0),
        "random_forest": RandomForestRegressor(n_estimators=EXPERIMENT["n_estimators"], random_state=EXPERIMENT["random_state"], min_samples_leaf=5),
    }
    for period in test_months:
        test = data[data["date"].dt.to_period("M") == period].copy()
        train = data[data["date"] < test["date"].min()].copy()
        train = embargo_train(train, test["date"])
        if len(train) < EXPERIMENT["min_train_size"] or len(test) < EXPERIMENT["min_test_size"]:
            logger.warning("Skipping fold %s because train/test is too small", period)
            continue
        scaler = StandardScaler().fit(train[active_features])
        X_train = scaler.transform(train[active_features])
        X_test = scaler.transform(test[active_features])
        y_train = train[EXPERIMENT["target"]]
        y_test = test[EXPERIMENT["target"]]
        for model_name in EXPERIMENT["models"]:
            model = models[model_name]
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            fold = {"period": str(period), "model": model_name, "N": len(test), "rmse": np.sqrt(mean_squared_error(y_test, pred)), "mae": mean_absolute_error(y_test, pred), "r2": r2_score(y_test, pred)}
            folds.append(fold)
            preds.append(pd.DataFrame({"date": test["date"], "model": model_name, "prediction": pred, "target": y_test.values}))
    pred_df = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame(columns=["date", "model", "prediction", "target"])
    return pd.DataFrame(folds), pred_df

walk_metrics, predictions = walk_forward(model_data, feature_cols)
write_table(walk_metrics, TABLE_DIR / "Table_9_ml_walk_forward.csv")
print(walk_metrics.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
if not walk_metrics.empty:
    sns.lineplot(data=walk_metrics, x="period", y="rmse", hue="model", palette=MODEL_COLORS, marker="o", ax=ax)
ax.axhline(0, color="black", linestyle="--", lw=0.8)
ax.set_title("Walk-Forward RMSE by Fold", fontweight="bold")
ax.set_xlabel("Test period")
ax.set_ylabel("RMSE (return decimal)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_9_walk_forward_rmse.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 10. Feature Ablation

Rerun the walk-forward loop with one feature block at a time and compare each block against controls.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Ablation asks which data family earns its complexity. This is especially important when combining market, fundamental, and alternative evidence.


In [ ]:
def features_for_block(block: str) -> list[str]:
    if block == "controls":
        return [col for col in feature_cols if col.startswith("control")]
    prefix = block.split("_")[0]
    cols = [col for col in feature_cols if prefix in col or col.startswith(block)]
    controls = [col for col in feature_cols if col.startswith("control")]
    return sorted(set(controls + cols))

ablation_rows = []
if EXPERIMENT["run_ablation"]:
    baseline_features = features_for_block("controls") or feature_cols[:1]
    baseline_metrics, _ = walk_forward(model_data, baseline_features)
    baseline_rmse = baseline_metrics["rmse"].mean() if not baseline_metrics.empty else np.nan
    for block in EXPERIMENT["feature_blocks"]:
        active = features_for_block(block) or baseline_features
        block_metrics, _ = walk_forward(model_data, active)
        rmse = block_metrics["rmse"].mean() if not block_metrics.empty else np.nan
        ablation_rows.append({"block": block, "N": int(block_metrics["N"].sum()) if not block_metrics.empty else 0, "rmse": rmse, "delta_metric_vs_controls": baseline_rmse - rmse})
ablation_table = pd.DataFrame(ablation_rows).sort_values("delta_metric_vs_controls", ascending=False)
write_table(ablation_table, TABLE_DIR / "Table_ablation.csv")
print(ablation_table.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if not ablation_table.empty:
    axes[0].barh(ablation_table["block"], ablation_table["delta_metric_vs_controls"], color=COLORS["primary"])
    cumulative = ablation_table["delta_metric_vs_controls"].fillna(0).cumsum()
    axes[1].plot(ablation_table["block"], cumulative, color=COLORS["accent"], marker="o")
for ax in axes:
    ax.axhline(0, color="black", linestyle="--", lw=0.8)
    ax.set_xlabel("Delta RMSE improvement (decimal)")
    ax.set_ylabel("Feature block")
axes[0].set_title("Feature Ablation", fontweight="bold")
axes[1].set_title("Cumulative Best-First Ablation", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_10_feature_ablation.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 11. Backtest / Strategy Evaluation

Convert predictions or scores into a simple cost-aware strategy evaluation.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

A predictive edge is not automatically tradable. Transaction costs and turnover determine whether statistical signal survives implementation.


In [ ]:
strategy = predictions.copy()
if not strategy.empty:
    strategy = strategy[strategy["model"] == strategy["prediction"].groupby(predictions["model"]).count().idxmax()].copy()
    upper = strategy["prediction"].quantile(EXPERIMENT["top_quantile"])
    lower = strategy["prediction"].quantile(EXPERIMENT["bottom_quantile"])
    strategy["position"] = np.select([strategy["prediction"] >= upper, strategy["prediction"] <= lower], [1, -1], default=0)
    strategy["turnover"] = strategy["position"].diff().abs().fillna(strategy["position"].abs())
    strategy["strategy_return"] = strategy["position"] * strategy["target"] - strategy["turnover"] * EXPERIMENT["cost_bps"] / 10000
    strategy["cum_strategy_return"] = (1 + strategy["strategy_return"]).cumprod() - 1
else:
    strategy = pd.DataFrame(columns=["date", "position", "strategy_return", "cum_strategy_return"])
backtest_table = strategy[["date", "position", "strategy_return", "cum_strategy_return"]].copy() if not strategy.empty else strategy
write_table(backtest_table, TABLE_DIR / "Table_11_backtest_strategy_evaluation.csv")
print(backtest_table.tail(20).round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
if not strategy.empty:
    ax.plot(strategy["date"], strategy["cum_strategy_return"], color=COLORS["primary"], label="strategy")
ax.axhline(0, color="black", linestyle="--", lw=0.8)
ax.set_title("Cost-Aware Strategy Evaluation", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative return (decimal)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_11_backtest_strategy_evaluation.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 12. Interpretability

Summarize model coefficients or feature importances to explain what drove predictions.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Interpretability connects model behavior back to economic intuition. This lowers the risk of deploying a spurious black-box pattern.


In [ ]:
importance_rows = []
if feature_cols:
    ridge = Ridge(alpha=1.0).fit(StandardScaler().fit_transform(model_data[feature_cols]), model_data[EXPERIMENT["target"]])
    for feature, coef in zip(feature_cols, ridge.coef_):
        importance_rows.append({"feature": feature, "N": len(model_data), "importance": abs(coef), "signed_effect": coef})
importance_table = pd.DataFrame(importance_rows).sort_values("importance", ascending=False)
write_table(importance_table, TABLE_DIR / "Table_12_interpretability.csv")
print(importance_table.head(20).round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
plot_imp = importance_table.head(15).sort_values("importance")
if not plot_imp.empty:
    ax.barh(plot_imp["feature"], plot_imp["importance"], color=COLORS["purple"])
ax.set_title("Model Interpretability", fontweight="bold")
ax.set_xlabel("Absolute standardized coefficient (decimal)")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_12_interpretability.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 13. Robustness Checks

Run subperiod, placebo, and sensitivity checks, then save the robustness table.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

Robustness checks test whether the result is stable or merely fitted to one sample slice. Placebos should collapse toward baseline if the signal is real.


In [ ]:
robust_rows = []
if not model_data.empty:
    median_date = model_data["date"].median()
    for regime_name, subset in {"early": model_data[model_data["date"] <= median_date], "late": model_data[model_data["date"] > median_date]}.items():
        if len(subset) >= EXPERIMENT["min_train_size"]:
            corr = subset[feature_cols[0]].corr(subset[EXPERIMENT["target"]]) if feature_cols else np.nan
            robust_rows.append({"test": "subperiod", "regime": regime_name, "parameter": "median_date", "value": str(median_date.date()), "N": len(subset), "metric": corr})
    placebo = model_data.copy()
    placebo[EXPERIMENT["target"]] = np.random.default_rng(EXPERIMENT["random_state"]).permutation(placebo[EXPERIMENT["target"]].values)
    placebo_corr = placebo[feature_cols[0]].corr(placebo[EXPERIMENT["target"]]) if feature_cols else np.nan
    robust_rows.append({"test": "placebo", "regime": "target_permutation", "parameter": "random_state", "value": EXPERIMENT["random_state"], "N": len(placebo), "metric": placebo_corr})
    for cost in [0.0, EXPERIMENT["cost_bps"], EXPERIMENT["cost_bps"] * 2]:
        metric = strategy["strategy_return"].mean() - cost / 10000 if not strategy.empty else np.nan
        robust_rows.append({"test": "sensitivity", "regime": "cost", "parameter": "cost_bps", "value": cost, "N": len(strategy), "metric": metric})
robust_table = pd.DataFrame(robust_rows)
write_table(robust_table, TABLE_DIR / "Table_VII_robustness.csv")
print(robust_table.round(4).to_string(index=False))

heat = robust_table[robust_table["test"] == "sensitivity"].copy()
fig, ax = plt.subplots(figsize=(10, 5))
if not heat.empty:
    heat_matrix = heat.pivot_table(index="parameter", columns="value", values="metric")
    sns.heatmap(heat_matrix, annot=True, fmt=".4f", cmap="viridis", ax=ax)
ax.set_title("Robustness Sensitivity Heatmap", fontweight="bold")
ax.set_xlabel("Parameter value")
ax.set_ylabel("Parameter")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Figure_13_robustness_sensitivity.png", dpi=EXPERIMENT["figure_dpi"])
plt.show()


## 14. Final Summary

List generated tables and figures so the research artifact is auditable.

| Cell | What it does | Output |
|---|---|---|
| Section code | Runs the section workflow using EXPERIMENT parameters | Saved table and/or figure |

$$
r_{t+h}=\frac{P_{t+h}}{P_t}-1,\quad z_t=\frac{x_t-\mu_{train}}{\sigma_{train}}\n$$

The final inventory makes the notebook easy to review, rerun, and compare across projects.


In [ ]:
import os
from pathlib import Path
tables  = sorted(Path("output/tables").glob("*.csv"))
figures = sorted(Path("output/figures").glob("*.png"))
print(f"✅ EXPERIMENT COMPLETE")
print(f"   Tables : {len(tables)}")
print(f"   Figures: {len(figures)}")
for f in tables:  print(f"   📋 {f.name}")
for f in figures: print(f"   📊 {f.name}")
